# Rotina de alimentação Oracle por YAML

Rotina destrutiva de desenvolvimento: valida todos os dados antes da conexão, limpa as tabelas na ordem de dependência e realimenta os catálogos definidos em YAML.

A flag `ATUALIZAR_AVS_RCM` controla a carga opcional de `AVS_SELD`, `RCM_VLDD` e `RCM_VRS`: `False` mantém as tabelas vazias; `True` carrega o fixture de cenários de vinculação.


## 01. Pré-validação local dos YAMLs


In [ ]:
from datetime import datetime
from pathlib import Path
from traceback import format_exc
import json
import re

import yaml


ATUALIZAR_AVS_RCM = False


class UniqueKeyLoader(yaml.SafeLoader):
    pass


def construir_mapeamento_unico(loader, node, deep=False):
    loader.flatten_mapping(node)
    resultado = {}
    for chave_node, valor_node in node.value:
        chave = loader.construct_object(chave_node, deep=deep)
        if chave in resultado:
            raise ValueError(f"Chave YAML duplicada: {chave!r}")
        resultado[chave] = loader.construct_object(valor_node, deep=deep)
    return resultado


UniqueKeyLoader.add_constructor(
    yaml.resolver.BaseResolver.DEFAULT_MAPPING_TAG,
    construir_mapeamento_unico,
)


def campo(tipo, nullable=False, tamanho_maximo=None):
    return {
        "tipo": tipo,
        "nullable": nullable,
        "tamanho_maximo": tamanho_maximo,
    }


CONTRATOS = {
    "PMPT_TCN": {
        "NM_PMPT_PDRO": campo("texto", tamanho_maximo=100),
        "NR_ETP_PSCL": campo("inteiro"),
        "NR_VRS_ATU": campo("inteiro"),
        "TX_FUC_OPRL_PMPT": campo("texto"),
        "TX_DTZ_ORTR_PMPT": campo("texto"),
        "TX_PDRO_DFND_PMPT": campo("texto"),
        "QT_VRV_TTL": campo("inteiro", nullable=True),
    },
    "PBCO_CADD": {
        "NM_PBCO_PDRO": campo("texto", tamanho_maximo=500),
        "TX_DCR_DETD_PBCO": campo("texto"),
        "TX_NCDD_RCNL_PBCO": campo("texto"),
        "TX_NCDD_EMOC_PBCO": campo("texto"),
        "TX_TRM_OBG_PBCO": campo("texto", nullable=True),
        "TX_TRM_N_PMT_PBCO": campo("texto", nullable=True),
        "NM_DB": campo("texto", tamanho_maximo=100),
        "NM_TAB_DB": campo("texto", tamanho_maximo=100),
        "NM_COL_TAB": campo("texto", nullable=True, tamanho_maximo=100),
        "TS_CAD": campo("timestamp"),
        "CD_EST_PBCO": campo("inteiro"),
        "CD_TIP_TAB": campo("inteiro"),
        "CD_USU_RSP_CAD": campo("texto", tamanho_maximo=8),
    },
    "SGT_CADD": {
        "TX_REG_EXNO_SGT": campo("texto", nullable=True, tamanho_maximo=100),
        "NM_SGT_PDRO": campo("texto", tamanho_maximo=500),
        "TX_DCR_DETD_SGT": campo("texto"),
        "TX_BNF_RCNL_SGT": campo("texto"),
        "TX_BNF_EMOC_SGT": campo("texto"),
        "TX_TRM_OBG_SGT": campo("texto", nullable=True),
        "TX_TRM_N_PMT_SGT": campo("texto", nullable=True),
        "TS_CAD_SGT": campo("timestamp"),
        "CD_EST_SGT": campo("inteiro"),
        "CD_USU_RSP_CAD_SGT": campo("texto", tamanho_maximo=8),
    },
    "TND_CGTV_TCN": {
        "NM_TND_PDRO": campo("texto", tamanho_maximo=500),
        "TX_DCR_DETD_TND": campo("texto"),
        "TX_FUC_OPRL_TND": campo("texto"),
        "TX_RCPO_PDRO_TND": campo("texto", nullable=True),
    },
    "APSC_TCN": {
        "NM_APSC_PDRO": campo("texto", tamanho_maximo=500),
        "TX_FUC_OPRL_APSC": campo("texto"),
        "TX_DCR_DETD_APSC": campo("texto"),
        "TX_RCPO_PDRO_APSC": campo("texto", nullable=True),
    },
    "CNR_TCN": {
        "NM_CNR_PDRO": campo("texto", tamanho_maximo=500),
        "TX_FUC_OPRL_CNR": campo("texto"),
        "TX_DCR_DETD_CNR": campo("texto"),
        "TX_RCPO_PDRO_CNR": campo("texto", nullable=True),
    },
    "PROJ_CADD": {
        "NM_PROJ_CMT": campo("texto", tamanho_maximo=500),
        "TX_DCR_DETD": campo("texto", tamanho_maximo=750),
        "TS_CAD": campo("timestamp"),
        "CD_EST_PROJ": campo("inteiro"),
        "CD_USU_RSP_CAD_PROJ": campo("texto", tamanho_maximo=8),
    },
}

CONTRATOS_VINCULACAO = {
    "AVS_SELD": {
        "NR_AVS_SELD": campo("inteiro"),
        "CD_EXNO_CRIC": campo("texto", tamanho_maximo=36),
        "CD_EXNO_AVS": campo("texto", tamanho_maximo=36),
        "TX_TIT_PDRO_AVS": campo("texto"),
        "TX_STIT_PDRO_AVS": campo("texto"),
        "TX_ACMT_PDRO_AVS": campo("texto"),
        "JS_AVS_PDRO": campo("json"),
        "TS_CAD_AVS": campo("timestamp"),
        "CD_USU_RSP_CAD": campo("texto", tamanho_maximo=8),
    },
    "RCM_VLDD": {
        "NR_IDFR_RCM": campo("inteiro"),
        "NR_VLDO_VRS": campo("inteiro"),
        "NR_VRS_PRPT": campo("inteiro"),
        "NR_PRI_SELD": campo("inteiro"),
        "TS_ULT_ATL": campo("timestamp"),
        "NM_RCM_PDRO": campo("texto", tamanho_maximo=500),
        "TX_DCR_DETD": campo("texto", tamanho_maximo=750),
        "CD_EST_RCM": campo("inteiro"),
    },
    "RCM_VRS": {
        "NR_VRS_PRPT": campo("inteiro"),
        "TX_DCR_DETD": campo("texto", tamanho_maximo=750),
        "DT_PRPT_INC_RCM": campo("data"),
        "DT_PRPT_FIM_RCM": campo("data"),
        "TX_PRPT_PRM_HDR_API": campo("json"),
        "TX_MTV_AVLC": campo("texto", nullable=True, tamanho_maximo=1000),
        "TS_ULT_ALT": campo("timestamp"),
        "CD_USU_RSP_CAD": campo("texto", tamanho_maximo=8),
        "CD_EST_RCM_VRS": campo("inteiro"),
    },
}

REFERENCIAS_VINCULACAO = {
    "AVS_SELD": {"publico", "sugestao", "apresentacao", "cenario", "tendencia"},
    "RCM_VLDD": {"projeto"},
    "RCM_VRS": {"recomendacao", "aviso"},
}
CONTAGENS_VINCULACAO = {"AVS_SELD": 15, "RCM_VLDD": 11, "RCM_VRS": 15}
DOMINIOS_VINCULACAO = {"CD_EST_RCM": set(range(1, 6)), "CD_EST_RCM_VRS": {1, 2, 3}}

COLUNAS_IDENTITY = {
    "NR_PMPT_IDFR",
    "NR_IDFR_PBCO",
    "NR_IDFR_SGT",
    "NR_IDFR_TND",
    "NR_IDFR_APSC",
    "NR_IDFR_CNR",
    "NR_IDFR_PROJ",
}

NOMES_POR_TABELA = {
    "PMPT_TCN": "NM_PMPT_PDRO",
    "PBCO_CADD": "NM_PBCO_PDRO",
    "SGT_CADD": "NM_SGT_PDRO",
    "TND_CGTV_TCN": "NM_TND_PDRO",
    "APSC_TCN": "NM_APSC_PDRO",
    "CNR_TCN": "NM_CNR_PDRO",
    "PROJ_CADD": "NM_PROJ_CMT",
}


def localizar_raiz() -> Path:
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / "src" / "utils" / "gerenciador_sessao_spark_local.py").is_file():
            return base
    raise FileNotFoundError("Raiz do projeto feed_oracle não encontrada.")


def ler_yaml(caminho: Path) -> dict:
    with caminho.open("r", encoding="utf-8") as arquivo:
        documento = yaml.load(arquivo, Loader=UniqueKeyLoader)
    if not isinstance(documento, dict):
        raise TypeError(f"{caminho}: a raiz do YAML precisa ser um objeto.")
    return documento


def validar_valor(caminho, coluna, valor, definicao):
    if valor is None:
        if not definicao["nullable"]:
            raise ValueError(f"{caminho}: {coluna} não pode ser nulo.")
        return

    tipo = definicao["tipo"]
    if tipo == "texto":
        if not isinstance(valor, str):
            raise TypeError(f"{caminho}: {coluna} precisa ser texto.")
        limite = definicao["tamanho_maximo"]
        if limite is not None and len(valor) > limite:
            raise ValueError(
                f"{caminho}: {coluna} possui {len(valor)} caracteres; limite={limite}."
            )
    elif tipo == "inteiro":
        if not isinstance(valor, int) or isinstance(valor, bool):
            raise TypeError(f"{caminho}: {coluna} precisa ser inteiro.")
    elif tipo == "timestamp":
        if valor != "__NOW__":
            raise ValueError(f"{caminho}: {coluna} precisa usar __NOW__.")
    elif tipo == "data":
        if not isinstance(valor, str):
            raise TypeError(f"{caminho}: {coluna} precisa usar data ISO como texto.")
        try:
            datetime.strptime(valor, "%Y-%m-%d")
        except ValueError as exc:
            raise ValueError(f"{caminho}: {coluna} precisa usar YYYY-MM-DD.") from exc
    elif tipo == "json":
        if not isinstance(valor, str):
            raise TypeError(f"{caminho}: {coluna} precisa ser texto JSON.")
        try:
            json.loads(valor)
        except json.JSONDecodeError as exc:
            raise ValueError(f"{caminho}: {coluna} contém JSON inválido.") from exc


def validar_fixture_vinculacao(caminho, documento, registros_catalogo):
    chaves_raiz = set(documento)
    if chaves_raiz != {"versao", "metadados", "tabelas"}:
        raise ValueError(
            f"{caminho}: chaves esperadas versao/metadados/tabelas; "
            f"encontradas={sorted(chaves_raiz)}."
        )
    if documento["versao"] != 1:
        raise ValueError(f"{caminho}: versao precisa ser 1.")
    if not isinstance(documento["metadados"], dict):
        raise TypeError(f"{caminho}: metadados precisa ser um objeto.")
    tabelas = documento["tabelas"]
    if not isinstance(tabelas, dict) or set(tabelas) != set(CONTRATOS_VINCULACAO):
        raise ValueError(
            f"{caminho}: tabelas precisa conter exatamente "
            f"{sorted(CONTRATOS_VINCULACAO)}."
        )

    aliases_avs = {}
    aliases_rcm = {}
    ids_avs = set()
    ids_rcm = set()
    chaves_vrs = set()

    for tabela, quantidade_esperada in CONTAGENS_VINCULACAO.items():
        itens = tabelas[tabela]
        if not isinstance(itens, list):
            raise TypeError(f"{caminho}: {tabela} precisa ser uma lista.")
        if len(itens) != quantidade_esperada:
            raise ValueError(
                f"{caminho}: {tabela} possui {len(itens)} registros; "
                f"esperado={quantidade_esperada}."
            )

        for indice, item in enumerate(itens, start=1):
            contexto = f"{caminho}:{tabela}[{indice}]"
            if not isinstance(item, dict):
                raise TypeError(f"{contexto}: item precisa ser um objeto.")
            chaves_item = {"referencias", "registro"}
            if tabela != "RCM_VRS":
                chaves_item.add("alias")
            if set(item) != chaves_item:
                raise ValueError(f"{contexto}: chaves inválidas; esperado={sorted(chaves_item)}.")

            referencias = item["referencias"]
            registro = item["registro"]
            if not isinstance(referencias, dict) or set(referencias) != REFERENCIAS_VINCULACAO[tabela]:
                raise ValueError(
                    f"{contexto}: referências inválidas; "
                    f"esperado={sorted(REFERENCIAS_VINCULACAO[tabela])}."
                )
            if any(not isinstance(valor, str) or not valor.strip() for valor in referencias.values()):
                raise ValueError(f"{contexto}: toda referência precisa ser texto não vazio.")
            if not isinstance(registro, dict) or set(registro) != set(CONTRATOS_VINCULACAO[tabela]):
                recebidas = set(registro) if isinstance(registro, dict) else set()
                esperadas = set(CONTRATOS_VINCULACAO[tabela])
                raise ValueError(
                    f"{contexto}: contrato divergente; faltantes={sorted(esperadas - recebidas)}; "
                    f"extras={sorted(recebidas - esperadas)}."
                )
            for coluna, definicao in CONTRATOS_VINCULACAO[tabela].items():
                validar_valor(contexto, coluna, registro[coluna], definicao)

            if tabela != "RCM_VRS":
                alias = item["alias"]
                if not isinstance(alias, str) or not alias.strip():
                    raise ValueError(f"{contexto}: alias precisa ser texto não vazio.")
                destino = aliases_avs if tabela == "AVS_SELD" else aliases_rcm
                if alias in destino:
                    raise ValueError(f"{contexto}: alias duplicado: {alias}.")
                destino[alias] = item

            if tabela == "AVS_SELD":
                nr_avs = registro["NR_AVS_SELD"]
                if nr_avs in ids_avs:
                    raise ValueError(f"{contexto}: NR_AVS_SELD duplicado: {nr_avs}.")
                ids_avs.add(nr_avs)
            elif tabela == "RCM_VLDD":
                nr_rcm = registro["NR_IDFR_RCM"]
                if nr_rcm in ids_rcm:
                    raise ValueError(f"{contexto}: NR_IDFR_RCM duplicado: {nr_rcm}.")
                ids_rcm.add(nr_rcm)
                if registro["CD_EST_RCM"] not in DOMINIOS_VINCULACAO["CD_EST_RCM"]:
                    raise ValueError(f"{contexto}: CD_EST_RCM fora do domínio.")
                if registro["NR_VLDO_VRS"] > registro["NR_VRS_PRPT"]:
                    raise ValueError(f"{contexto}: versão válida maior que a proposta.")
            else:
                chave = (referencias["recomendacao"], registro["NR_VRS_PRPT"])
                if chave in chaves_vrs:
                    raise ValueError(f"{contexto}: versão duplicada para recomendação: {chave}.")
                chaves_vrs.add(chave)
                if registro["CD_EST_RCM_VRS"] not in DOMINIOS_VINCULACAO["CD_EST_RCM_VRS"]:
                    raise ValueError(f"{contexto}: CD_EST_RCM_VRS fora do domínio.")
                inicio = datetime.strptime(registro["DT_PRPT_INC_RCM"], "%Y-%m-%d")
                fim = datetime.strptime(registro["DT_PRPT_FIM_RCM"], "%Y-%m-%d")
                if inicio > fim:
                    raise ValueError(f"{contexto}: data inicial posterior à final.")

    nomes_catalogo = {
        "publico": {r["NM_PBCO_PDRO"] for r in registros_catalogo["PBCO_CADD"]},
        "sugestao": {r["NM_SGT_PDRO"] for r in registros_catalogo["SGT_CADD"]},
        "apresentacao": {r["NM_APSC_PDRO"] for r in registros_catalogo["APSC_TCN"]},
        "cenario": {r["NM_CNR_PDRO"] for r in registros_catalogo["CNR_TCN"]},
        "tendencia": {r["NM_TND_PDRO"] for r in registros_catalogo["TND_CGTV_TCN"]},
        "projeto": {r["NM_PROJ_CMT"] for r in registros_catalogo["PROJ_CADD"]},
    }
    for tabela in ("AVS_SELD", "RCM_VLDD"):
        for item in tabelas[tabela]:
            for tipo_ref, valor_ref in item["referencias"].items():
                if valor_ref not in nomes_catalogo[tipo_ref]:
                    raise ValueError(f"{caminho}: referência inexistente em {tipo_ref}: {valor_ref!r}.")

    versoes_por_rcm = {}
    for item in tabelas["RCM_VRS"]:
        alias_rcm = item["referencias"]["recomendacao"]
        alias_avs = item["referencias"]["aviso"]
        if alias_rcm not in aliases_rcm:
            raise ValueError(f"{caminho}: recomendação inexistente: {alias_rcm}.")
        if alias_avs not in aliases_avs:
            raise ValueError(f"{caminho}: aviso inexistente: {alias_avs}.")
        versoes_por_rcm.setdefault(alias_rcm, {})[item["registro"]["NR_VRS_PRPT"]] = item

    for alias_rcm, item in aliases_rcm.items():
        registro = item["registro"]
        versoes = versoes_por_rcm.get(alias_rcm, {})
        proposta = registro["NR_VRS_PRPT"]
        valida = registro["NR_VLDO_VRS"]
        if proposta not in versoes:
            raise ValueError(f"{caminho}: {alias_rcm} não possui a versão proposta {proposta}.")
        if valida > 0:
            versao_valida = versoes.get(valida)
            if versao_valida is None or versao_valida["registro"]["CD_EST_RCM_VRS"] != 2:
                raise ValueError(f"{caminho}: {alias_rcm} não possui versão válida {valida} aprovada.")

    return documento


try:
    raiz_projeto = localizar_raiz()
    fontes = {
        "PMPT_TCN": sorted(raiz_projeto.glob("prompts/*/p1.yaml")),
        "PBCO_CADD": sorted((raiz_projeto / "publicos").glob("*.yaml")),
        "SGT_CADD": sorted((raiz_projeto / "produtos").glob("*.yaml")),
        "TND_CGTV_TCN": sorted((raiz_projeto / "tendencias").glob("*.yaml")),
        "APSC_TCN": sorted((raiz_projeto / "apresentacoes").glob("*.yaml")),
        "CNR_TCN": sorted((raiz_projeto / "cenarios").glob("*.yaml")),
        "PROJ_CADD": sorted((raiz_projeto / "projetos").glob("*.yaml")),
    }

    registros_por_tabela = {tabela: [] for tabela in CONTRATOS}
    fontes_por_tabela = {tabela: [] for tabela in CONTRATOS}

    for tabela_esperada, caminhos in fontes.items():
        if not caminhos:
            raise FileNotFoundError(f"Nenhum YAML encontrado para {tabela_esperada}.")

        nomes_encontrados = set()
        for caminho in caminhos:
            documento = ler_yaml(caminho)
            chaves_raiz = set(documento)
            if chaves_raiz != {"tabela", "registro", "metadados"}:
                raise ValueError(
                    f"{caminho}: chaves esperadas tabela/registro/metadados; "
                    f"encontradas={sorted(chaves_raiz)}."
                )
            if documento["tabela"] != tabela_esperada:
                raise ValueError(
                    f"{caminho}: tabela={documento['tabela']!r}; "
                    f"esperado={tabela_esperada!r}."
                )
            if not isinstance(documento["registro"], dict):
                raise TypeError(f"{caminho}: registro precisa ser um objeto.")
            if not isinstance(documento["metadados"], dict):
                raise TypeError(f"{caminho}: metadados precisa ser um objeto.")

            registro = documento["registro"]
            recebidas = set(registro)
            esperadas = set(CONTRATOS[tabela_esperada])
            if recebidas & COLUNAS_IDENTITY:
                raise ValueError(
                    f"{caminho}: coluna IDENTITY não pode ser informada: "
                    f"{sorted(recebidas & COLUNAS_IDENTITY)}."
                )
            if recebidas != esperadas:
                raise ValueError(
                    f"{caminho}: contrato de {tabela_esperada} divergente; "
                    f"faltantes={sorted(esperadas - recebidas)}; "
                    f"extras={sorted(recebidas - esperadas)}."
                )

            for coluna, definicao in CONTRATOS[tabela_esperada].items():
                validar_valor(caminho, coluna, registro[coluna], definicao)

            coluna_nome = NOMES_POR_TABELA[tabela_esperada]
            nome = registro[coluna_nome]
            if nome in nomes_encontrados:
                raise ValueError(
                    f"{tabela_esperada}: nome duplicado em {coluna_nome}: {nome!r}."
                )
            nomes_encontrados.add(nome)
            registros_por_tabela[tabela_esperada].append(registro)
            fontes_por_tabela[tabela_esperada].append(str(caminho.relative_to(raiz_projeto)))

    prompts = registros_por_tabela["PMPT_TCN"]
    etapas = {registro["NR_ETP_PSCL"] for registro in prompts}
    if etapas != set(range(1, 8)) or len(prompts) != 7:
        raise ValueError("PMPT_TCN precisa conter exatamente um p1 para cada etapa de 1 a 7.")
    if any(registro["NR_VRS_ATU"] != 1 for registro in prompts):
        raise ValueError("Todos os prompts selecionados precisam declarar NR_VRS_ATU=1.")

    padrao_placeholder = re.compile(r"PH_[A-Z0-9_]+")
    for registro in prompts:
        quantidade = len(set(padrao_placeholder.findall(registro["TX_PDRO_DFND_PMPT"])))
        if registro["QT_VRV_TTL"] != quantidade:
            raise ValueError(
                f"{registro['NM_PMPT_PDRO']}: QT_VRV_TTL={registro['QT_VRV_TTL']}; "
                f"placeholders encontrados={quantidade}."
            )

    fixture_vinculacao = None
    if ATUALIZAR_AVS_RCM:
        caminho_fixture = raiz_projeto / "vinculacao" / "cenarios_vinculacao.yaml"
        if not caminho_fixture.is_file():
            raise FileNotFoundError(f"Fixture de vinculação não encontrada: {caminho_fixture}.")
        fixture_vinculacao = validar_fixture_vinculacao(
            caminho_fixture,
            ler_yaml(caminho_fixture),
            registros_por_tabela,
        )

    bundle = {
        "registros": registros_por_tabela,
        "fontes": fontes_por_tabela,
        "atualizar_avs_rcm": ATUALIZAR_AVS_RCM,
        "fixture_vinculacao": fixture_vinculacao,
    }
    print("Pré-validação YAML concluída antes da conexão Oracle:")
    for tabela, registros in registros_por_tabela.items():
        print(f"  {tabela}: {len(registros)} registros")
    if ATUALIZAR_AVS_RCM:
        for tabela, registros in fixture_vinculacao["tabelas"].items():
            print(f"  {tabela}: {len(registros)} registros (carga opcional ativa)")
    else:
        print("  AVS_SELD/RCM_VLDD/RCM_VRS: carga opcional desativada")
    print("  prompts ignorados pelo reload: todos os arquivos diferentes de p1.yaml")

except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise


## 02. Sessão Spark de desenvolvimento


In [ ]:
try:
    from src.utils.gerenciador_sessao_spark_local import (
        GerenciadorSessaoSpark,
        ler_variavel_ambiente_local,
    )

    ambiente = ler_variavel_ambiente_local("AMBIENTE").upper()
    if ambiente != "MODELAGEM":
        raise EnvironmentError(
            "Reload destrutivo bloqueado: AMBIENTE precisa ser MODELAGEM."
        )

    gerenciador_spark = GerenciadorSessaoSpark(
        nome_sessao="feed-oracle-yaml",
        adicionar_variaveis={
            "DOMINIO": "t2i",
            "SANDBOX": "t2i2016",
            "AMBIENTE": ambiente,
        },
        nome_arquivo_env_modelagem="desenv.env",
        exibir_configuracao=False,
        ativar_logs=True,
    )

    spark = gerenciador_spark.criar_sessao_spark(
        db2=True,
        driver_memory="16g",
        jars=[
            "/dados/shared/bin/ojdbc8.jar",
        ],
        spark_conf={
            "spark.driver.memoryOverhead": "8g",
        },
    )
    spark.send_to_spark(bundle)
    print("Bundle YAML enviado para a sessão Spark.")

    get_ipython().display_formatter.formatters["text/plain"].for_type(
        __import__("ipywidgets").Widget,
        lambda *args, **kwargs: None,
    )

except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise


## 03. Utilitários remotos e cliente Oracle


In [ ]:
try:
    %run ./src/utils/gerenciador_sessao_spark_remoto.ipynb
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise


In [ ]:
%%spark

import os
from datetime import datetime

import pyspark.sql.types as T


ambiente = ler_variavel_ambiente_spark("AMBIENTE").upper()
if ambiente != "MODELAGEM":
    raise EnvironmentError(
        "Reload destrutivo bloqueado: a sessão Spark não está em MODELAGEM."
    )

registros_por_tabela = bundle["registros"]
fontes_por_tabela = bundle["fontes"]
atualizar_avs_rcm = bool(bundle["atualizar_avs_rcm"])
fixture_vinculacao = bundle["fixture_vinculacao"]

cliente_oracle = criar_cliente_oracle_spark(env=dict(os.environ))
oracle_schema = cliente_oracle.schema.upper()

print(f"Ambiente validado: {ambiente}")
print(f"Schema Oracle de destino: {oracle_schema}")
print(f"Carga opcional AVS/RCM ativa: {atualizar_avs_rcm}")


## 04. Contratos e ordens de dependência


In [ ]:
%%spark

def coluna(nome, tipo, nullable=False):
    return {
        "nome": nome,
        "tipo": tipo,
        "nullable": nullable,
    }


TABLE_SPECS = {
    "PMPT_TCN": [
        coluna("NM_PMPT_PDRO", T.StringType()),
        coluna("NR_ETP_PSCL", T.IntegerType()),
        coluna("NR_VRS_ATU", T.IntegerType()),
        coluna("TX_FUC_OPRL_PMPT", T.StringType()),
        coluna("TX_DTZ_ORTR_PMPT", T.StringType()),
        coluna("TX_PDRO_DFND_PMPT", T.StringType()),
        coluna("QT_VRV_TTL", T.IntegerType(), nullable=True),
    ],
    "PBCO_CADD": [
        coluna("NM_PBCO_PDRO", T.StringType()),
        coluna("TX_DCR_DETD_PBCO", T.StringType()),
        coluna("TX_NCDD_RCNL_PBCO", T.StringType()),
        coluna("TX_NCDD_EMOC_PBCO", T.StringType()),
        coluna("TX_TRM_OBG_PBCO", T.StringType(), nullable=True),
        coluna("TX_TRM_N_PMT_PBCO", T.StringType(), nullable=True),
        coluna("NM_DB", T.StringType()),
        coluna("NM_TAB_DB", T.StringType()),
        coluna("NM_COL_TAB", T.StringType(), nullable=True),
        coluna("TS_CAD", T.TimestampType()),
        coluna("CD_EST_PBCO", T.IntegerType()),
        coluna("CD_TIP_TAB", T.IntegerType()),
        coluna("CD_USU_RSP_CAD", T.StringType()),
    ],
    "SGT_CADD": [
        coluna("TX_REG_EXNO_SGT", T.StringType(), nullable=True),
        coluna("NM_SGT_PDRO", T.StringType()),
        coluna("TX_DCR_DETD_SGT", T.StringType()),
        coluna("TX_BNF_RCNL_SGT", T.StringType()),
        coluna("TX_BNF_EMOC_SGT", T.StringType()),
        coluna("TX_TRM_OBG_SGT", T.StringType(), nullable=True),
        coluna("TX_TRM_N_PMT_SGT", T.StringType(), nullable=True),
        coluna("TS_CAD_SGT", T.TimestampType()),
        coluna("CD_EST_SGT", T.IntegerType()),
        coluna("CD_USU_RSP_CAD_SGT", T.StringType()),
    ],
    "TND_CGTV_TCN": [
        coluna("NM_TND_PDRO", T.StringType()),
        coluna("TX_DCR_DETD_TND", T.StringType()),
        coluna("TX_FUC_OPRL_TND", T.StringType()),
        coluna("TX_RCPO_PDRO_TND", T.StringType(), nullable=True),
    ],
    "APSC_TCN": [
        coluna("NM_APSC_PDRO", T.StringType()),
        coluna("TX_FUC_OPRL_APSC", T.StringType()),
        coluna("TX_DCR_DETD_APSC", T.StringType()),
        coluna("TX_RCPO_PDRO_APSC", T.StringType(), nullable=True),
    ],
    "CNR_TCN": [
        coluna("NM_CNR_PDRO", T.StringType()),
        coluna("TX_FUC_OPRL_CNR", T.StringType()),
        coluna("TX_DCR_DETD_CNR", T.StringType()),
        coluna("TX_RCPO_PDRO_CNR", T.StringType(), nullable=True),
    ],
    "PROJ_CADD": [
        coluna("NM_PROJ_CMT", T.StringType()),
        coluna("TX_DCR_DETD", T.StringType()),
        coluna("TS_CAD", T.TimestampType()),
        coluna("CD_EST_PROJ", T.IntegerType()),
        coluna("CD_USU_RSP_CAD_PROJ", T.StringType()),
    ],
    "AVS_SELD": [
        coluna("NR_AVS_SELD", T.LongType()),
        coluna("NR_IDFR_PBCO", T.LongType()),
        coluna("NR_IDFR_SGT", T.LongType()),
        coluna("CD_EXNO_CRIC", T.StringType()),
        coluna("CD_EXNO_AVS", T.StringType()),
        coluna("TX_TIT_PDRO_AVS", T.StringType()),
        coluna("TX_STIT_PDRO_AVS", T.StringType()),
        coluna("TX_ACMT_PDRO_AVS", T.StringType()),
        coluna("JS_AVS_PDRO", T.StringType()),
        coluna("TS_CAD_AVS", T.TimestampType()),
        coluna("CD_USU_RSP_CAD", T.StringType()),
        coluna("NR_IDFR_APSC", T.LongType()),
        coluna("NR_IDFR_CNR", T.LongType()),
        coluna("NR_IDFR_TND", T.LongType()),
    ],
    "RCM_VLDD": [
        coluna("NR_IDFR_RCM", T.LongType()),
        coluna("NR_IDFR_PROJ", T.LongType()),
        coluna("NR_VLDO_VRS", T.IntegerType()),
        coluna("NR_VRS_PRPT", T.IntegerType()),
        coluna("NR_PRI_SELD", T.IntegerType()),
        coluna("TS_ULT_ATL", T.TimestampType()),
        coluna("NM_RCM_PDRO", T.StringType()),
        coluna("TX_DCR_DETD", T.StringType()),
        coluna("CD_EST_RCM", T.IntegerType()),
    ],
    "RCM_VRS": [
        coluna("NR_IDFR_RCM", T.LongType()),
        coluna("NR_VRS_PRPT", T.IntegerType()),
        coluna("NR_AVS_SELD", T.LongType()),
        coluna("TX_DCR_DETD", T.StringType()),
        coluna("DT_PRPT_INC_RCM", T.DateType()),
        coluna("DT_PRPT_FIM_RCM", T.DateType()),
        coluna("TX_PRPT_PRM_HDR_API", T.StringType()),
        coluna("TX_MTV_AVLC", T.StringType(), nullable=True),
        coluna("TS_ULT_ALT", T.TimestampType()),
        coluna("NR_IDFR_PROJ", T.LongType()),
        coluna("CD_USU_RSP_CAD", T.StringType()),
        coluna("CD_EST_RCM_VRS", T.IntegerType()),
    ],
}

ORDEM_LIMPEZA = [
    "RCM_VRS",
    "RCM_VLDD",
    "AVS_SELD",
    "PROJ_CADD",
    "PBCO_CADD",
    "SGT_CADD",
    "TND_CGTV_TCN",
    "CNR_TCN",
    "APSC_TCN",
    "PMPT_TCN",
]

ORDEM_CARGA = [
    "PMPT_TCN",
    "PBCO_CADD",
    "SGT_CADD",
    "TND_CGTV_TCN",
    "APSC_TCN",
    "CNR_TCN",
    "PROJ_CADD",
]

ORDEM_CARGA_VINCULACAO = [
    "AVS_SELD",
    "RCM_VLDD",
    "RCM_VRS",
]

TABELAS_QUE_PERMANECEM_VAZIAS = [
    "AVS_SELD",
    "RCM_VLDD",
    "RCM_VRS",
]

ID_PROJETO_PADRAO = 1
# TEMPORÁRIO: remover esta flag, a função e sua chamada após regularizar a geração da PK.
NORMALIZAR_IDS_TND_TEMPORARIAMENTE = True

if set(registros_por_tabela) != set(ORDEM_CARGA):
    raise ValueError(
        "Bundle YAML não contém exatamente as tabelas previstas para carga: "
        f"{sorted(registros_por_tabela)}"
    )


## 05. Materialização dos DataFrames


In [ ]:
%%spark

def normalizar_valor(valor, tipo, agora):
    if isinstance(tipo, T.TimestampType) and valor == "__NOW__":
        return agora
    if isinstance(tipo, T.DateType) and isinstance(valor, str):
        return datetime.strptime(valor, "%Y-%m-%d").date()
    return valor


def criar_dataframe(tabela, registros, agora):
    especificacao = TABLE_SPECS[tabela]
    schema = T.StructType([
        T.StructField(item["nome"], item["tipo"], item["nullable"])
        for item in especificacao
    ])
    linhas = [
        tuple(
            normalizar_valor(registro[item["nome"]], item["tipo"], agora)
            for item in especificacao
        )
        for registro in registros
    ]
    return spark.createDataFrame(linhas, schema=schema)


agora_preflight = datetime.now()
dataframes = {}
for tabela in ORDEM_CARGA:
    registros = registros_por_tabela[tabela]
    dataframes[tabela] = criar_dataframe(tabela, registros, agora_preflight)
    if len(registros) == 0:
        raise ValueError(f"{tabela}: nenhuma linha foi materializada.")
    print(f"{tabela}: schema Spark validado para {len(registros)} registros")


## 06. Limpeza e reload Oracle


In [ ]:
%%spark

def contar_oracle(tabela):
    consulta = f"SELECT COUNT(1) AS QTD FROM {oracle_schema}.{tabela}"
    return int(cliente_oracle.run_select(consulta).collect()[0]["QTD"])


def reload_table(tabela, dataframe, quantidade_esperada):
    try:
        cliente_oracle.reload_dataframe(
            df=dataframe,
            table_name=tabela,
            batchsize=5000,
            num_partitions=1,
            use_truncate=False,
        )
    except Exception as exc:
        raise RuntimeError(f"Falha no reload de {tabela}: {exc}") from exc

    quantidade_oracle = contar_oracle(tabela)
    if quantidade_oracle != quantidade_esperada:
        raise RuntimeError(
            f"{tabela}: YAML={quantidade_esperada}; Oracle={quantidade_oracle}."
        )
    print(f"[OK] {tabela}: {quantidade_oracle} registros")


def normalizar_ids_tendencias_temporariamente():
    if not NORMALIZAR_IDS_TND_TEMPORARIAMENTE:
        print("Normalização temporária de TND_CGTV_TCN desativada.")
        return

    tabela = "TND_CGTV_TCN"
    coluna_id = "NR_IDFR_TND"
    coluna_nome = "NM_TND_PDRO"
    registros_yaml = registros_por_tabela[tabela]
    nomes_ordenados = [registro[coluna_nome] for registro in registros_yaml]

    if len(nomes_ordenados) != 10 or len(set(nomes_ordenados)) != 10:
        raise RuntimeError(
            "TND_CGTV_TCN precisa possuir exatamente 10 nomes únicos para "
            "normalizar NR_IDFR_TND de 1 a 10."
        )

    quantidade_avisos = contar_oracle("AVS_SELD")
    if quantidade_avisos != 0:
        raise RuntimeError(
            "AVS_SELD precisa estar vazia antes de normalizar NR_IDFR_TND; "
            f"registros={quantidade_avisos}."
        )

    ids_esperados = {
        nome: posicao
        for posicao, nome in enumerate(nomes_ordenados, start=1)
    }
    ids_temporarios = {nome: -identificador for nome, identificador in ids_esperados.items()}

    def consultar_ids_por_nome():
        linhas = cliente_oracle.run_select(
            f"SELECT {coluna_id}, {coluna_nome} FROM {oracle_schema}.{tabela}"
        ).collect()
        if len(linhas) != 10:
            raise RuntimeError(
                f"{tabela} deveria possuir 10 registros; encontrados={len(linhas)}."
            )

        resultado = {}
        for linha in linhas:
            nome = linha[coluna_nome]
            if nome in resultado:
                raise RuntimeError(f"{tabela}: nome duplicado no Oracle: {nome!r}.")
            resultado[nome] = int(linha[coluna_id])

        if set(resultado) != set(ids_esperados):
            raise RuntimeError(
                f"{tabela}: nomes Oracle divergentes dos YAMLs; "
                f"faltantes={sorted(set(ids_esperados) - set(resultado))}; "
                f"extras={sorted(set(resultado) - set(ids_esperados))}."
            )
        return resultado

    ids_atuais = consultar_ids_por_nome()
    if ids_atuais == ids_esperados:
        print("[OK] TND_CGTV_TCN: NR_IDFR_TND já normalizado de 1 a 10.")
        return

    for nome, identificador_atual in ids_atuais.items():
        if identificador_atual < 0 and identificador_atual != ids_temporarios[nome]:
            raise RuntimeError(
                f"{tabela}: ID temporário inesperado para {nome!r}: {identificador_atual}."
            )

    # TEMPORÁRIO: duas fases evitam colisão da PK ao reutilizar os IDs 1..10.
    for nome in nomes_ordenados:
        quantidade_atualizada = cliente_oracle.update(
            table_name=tabela,
            values={coluna_id: ids_temporarios[nome]},
            where={coluna_nome: nome},
        )
        if quantidade_atualizada != 1:
            raise RuntimeError(
                f"{tabela}: deveria mover uma linha para ID temporário; "
                f"nome={nome!r}; atualizadas={quantidade_atualizada}."
            )

    ids_intermediarios = consultar_ids_por_nome()
    if ids_intermediarios != ids_temporarios:
        raise RuntimeError(
            f"{tabela}: falha na fase temporária da normalização; "
            f"encontrados={ids_intermediarios}."
        )

    for nome in nomes_ordenados:
        quantidade_atualizada = cliente_oracle.update(
            table_name=tabela,
            values={coluna_id: ids_esperados[nome]},
            where={coluna_nome: nome},
        )
        if quantidade_atualizada != 1:
            raise RuntimeError(
                f"{tabela}: deveria normalizar uma linha; "
                f"nome={nome!r}; atualizadas={quantidade_atualizada}."
            )

    ids_finais = consultar_ids_por_nome()
    if ids_finais != ids_esperados:
        raise RuntimeError(
            f"{tabela}: falha ao normalizar NR_IDFR_TND; encontrados={ids_finais}."
        )

    print("[OK] TND_CGTV_TCN: NR_IDFR_TND normalizado temporariamente de 1 a 10.")


def normalizar_id_projeto():
    projetos = cliente_oracle.run_select(
        f"""
        SELECT NR_IDFR_PROJ, NM_PROJ_CMT
        FROM {oracle_schema}.PROJ_CADD
        """
    ).collect()

    if len(projetos) != 1:
        raise RuntimeError(
            f"PROJ_CADD deve possuir exatamente um projeto; encontrados={len(projetos)}."
        )

    for tabela_dependente in ("RCM_VLDD", "RCM_VRS"):
        quantidade = contar_oracle(tabela_dependente)
        if quantidade != 0:
            raise RuntimeError(
                f"{tabela_dependente} precisa estar vazia antes de alterar o ID do projeto; "
                f"registros={quantidade}."
            )

    id_atual = int(projetos[0]["NR_IDFR_PROJ"])
    if id_atual != ID_PROJETO_PADRAO:
        quantidade_atualizada = cliente_oracle.update(
            table_name="PROJ_CADD",
            values={"NR_IDFR_PROJ": ID_PROJETO_PADRAO},
            where={"NR_IDFR_PROJ": id_atual},
        )
        if quantidade_atualizada != 1:
            raise RuntimeError(
                f"PROJ_CADD deveria atualizar uma linha; atualizadas={quantidade_atualizada}."
            )

    projetos_atualizados = cliente_oracle.run_select(
        f"SELECT NR_IDFR_PROJ FROM {oracle_schema}.PROJ_CADD"
    ).collect()
    ids_encontrados = [
        int(projeto["NR_IDFR_PROJ"])
        for projeto in projetos_atualizados
    ]
    if ids_encontrados != [ID_PROJETO_PADRAO]:
        raise RuntimeError(
            f"Falha ao normalizar NR_IDFR_PROJ; encontrados={ids_encontrados}."
        )

    print(f"[OK] PROJ_CADD: NR_IDFR_PROJ={ID_PROJETO_PADRAO}")


def carregar_mapa_catalogo(tabela, coluna_nome, coluna_id):
    linhas = cliente_oracle.run_select(
        f"SELECT {coluna_nome}, {coluna_id} FROM {oracle_schema}.{tabela}"
    ).collect()
    mapa = {}
    for linha in linhas:
        nome = linha[coluna_nome]
        if nome in mapa:
            raise RuntimeError(f"{tabela}: nome duplicado ao resolver referência: {nome!r}.")
        mapa[nome] = int(linha[coluna_id])
    return mapa


def resolver_registros_vinculacao():
    mapas = {
        "publico": carregar_mapa_catalogo("PBCO_CADD", "NM_PBCO_PDRO", "NR_IDFR_PBCO"),
        "sugestao": carregar_mapa_catalogo("SGT_CADD", "NM_SGT_PDRO", "NR_IDFR_SGT"),
        "apresentacao": carregar_mapa_catalogo("APSC_TCN", "NM_APSC_PDRO", "NR_IDFR_APSC"),
        "cenario": carregar_mapa_catalogo("CNR_TCN", "NM_CNR_PDRO", "NR_IDFR_CNR"),
        "tendencia": carregar_mapa_catalogo("TND_CGTV_TCN", "NM_TND_PDRO", "NR_IDFR_TND"),
        "projeto": carregar_mapa_catalogo("PROJ_CADD", "NM_PROJ_CMT", "NR_IDFR_PROJ"),
    }

    def resolver(tipo, nome):
        if nome not in mapas[tipo]:
            raise RuntimeError(f"Referência Oracle não encontrada em {tipo}: {nome!r}.")
        return mapas[tipo][nome]

    tabelas = fixture_vinculacao["tabelas"]
    avisos = []
    avisos_por_alias = {}
    for item in tabelas["AVS_SELD"]:
        refs = item["referencias"]
        registro = dict(item["registro"])
        registro.update({
            "NR_IDFR_PBCO": resolver("publico", refs["publico"]),
            "NR_IDFR_SGT": resolver("sugestao", refs["sugestao"]),
            "NR_IDFR_APSC": resolver("apresentacao", refs["apresentacao"]),
            "NR_IDFR_CNR": resolver("cenario", refs["cenario"]),
            "NR_IDFR_TND": resolver("tendencia", refs["tendencia"]),
        })
        avisos.append(registro)
        avisos_por_alias[item["alias"]] = registro

    recomendacoes = []
    recomendacoes_por_alias = {}
    for item in tabelas["RCM_VLDD"]:
        registro = dict(item["registro"])
        registro["NR_IDFR_PROJ"] = resolver("projeto", item["referencias"]["projeto"])
        recomendacoes.append(registro)
        recomendacoes_por_alias[item["alias"]] = registro

    versoes = []
    for item in tabelas["RCM_VRS"]:
        refs = item["referencias"]
        recomendacao = recomendacoes_por_alias[refs["recomendacao"]]
        aviso = avisos_por_alias[refs["aviso"]]
        registro = dict(item["registro"])
        registro.update({
            "NR_IDFR_RCM": recomendacao["NR_IDFR_RCM"],
            "NR_IDFR_PROJ": recomendacao["NR_IDFR_PROJ"],
            "NR_AVS_SELD": aviso["NR_AVS_SELD"],
        })
        versoes.append(registro)

    return {
        "AVS_SELD": avisos,
        "RCM_VLDD": recomendacoes,
        "RCM_VRS": versoes,
    }


print("Limpando tabelas na ordem de dependência reversa...")
for tabela in ORDEM_LIMPEZA:
    try:
        print(f"  limpando {tabela}")
        cliente_oracle.limpar_tabela(
            table_name=tabela,
            batchsize=5000,
            use_truncate=False,
        )
    except Exception as exc:
        raise RuntimeError(f"Falha na limpeza de {tabela}: {exc}") from exc

for tabela in ORDEM_LIMPEZA:
    quantidade = contar_oracle(tabela)
    if quantidade != 0:
        raise RuntimeError(f"{tabela}: limpeza incompleta; registros={quantidade}.")

print("Realimentando tabelas a partir dos YAMLs...")
for tabela in ORDEM_CARGA:
    reload_table(
        tabela=tabela,
        dataframe=dataframes[tabela],
        quantidade_esperada=len(registros_por_tabela[tabela]),
    )
    if tabela == "TND_CGTV_TCN":
        normalizar_ids_tendencias_temporariamente()

normalizar_id_projeto()

registros_vinculacao_resolvidos = {}
if atualizar_avs_rcm:
    registros_vinculacao_resolvidos = resolver_registros_vinculacao()
    print("Realimentando tabelas opcionais de vinculação...")
    for tabela in ORDEM_CARGA_VINCULACAO:
        registros = registros_vinculacao_resolvidos[tabela]
        dataframe = criar_dataframe(tabela, registros, agora_preflight)
        reload_table(tabela, dataframe, len(registros))
else:
    print("Carga opcional desativada: AVS_SELD, RCM_VLDD e RCM_VRS permanecerão vazias.")

print("Reload Oracle concluído.")


## 07. Verificação final


In [ ]:
%%spark

inconsistencias = []
for tabela in ORDEM_CARGA:
    esperado = len(registros_por_tabela[tabela])
    encontrado = contar_oracle(tabela)
    status = "OK" if esperado == encontrado else "ERRO"
    print(f"[{status}] {tabela}: YAML={esperado} Oracle={encontrado}")
    if esperado != encontrado:
        inconsistencias.append((tabela, esperado, encontrado))

for tabela in TABELAS_QUE_PERMANECEM_VAZIAS:
    esperado = (
        len(fixture_vinculacao["tabelas"][tabela])
        if atualizar_avs_rcm
        else 0
    )
    encontrado = contar_oracle(tabela)
    status = "OK" if encontrado == esperado else "ERRO"
    print(f"[{status}] {tabela}: esperado={esperado} Oracle={encontrado}")
    if encontrado != esperado:
        inconsistencias.append((tabela, esperado, encontrado))

if atualizar_avs_rcm:
    consultas_integridade = {
        "AVS_SELD.NR_AVS_SELD_DUPLICADO": f"""
            SELECT COUNT(1) AS QTD FROM (
                SELECT NR_AVS_SELD
                FROM {oracle_schema}.AVS_SELD
                GROUP BY NR_AVS_SELD
                HAVING COUNT(1) > 1
            )
        """,
        "AVS_SELD.PK_DUPLICADA": f"""
            SELECT COUNT(1) AS QTD FROM (
                SELECT NR_IDFR_PBCO, NR_IDFR_SGT, NR_AVS_SELD
                FROM {oracle_schema}.AVS_SELD
                GROUP BY NR_IDFR_PBCO, NR_IDFR_SGT, NR_AVS_SELD
                HAVING COUNT(1) > 1
            )
        """,
        "RCM_VLDD.PK_DUPLICADA": f"""
            SELECT COUNT(1) AS QTD FROM (
                SELECT NR_IDFR_RCM, NR_IDFR_PROJ
                FROM {oracle_schema}.RCM_VLDD
                GROUP BY NR_IDFR_RCM, NR_IDFR_PROJ
                HAVING COUNT(1) > 1
            )
        """,
        "RCM_VRS.PK_DUPLICADA": f"""
            SELECT COUNT(1) AS QTD FROM (
                SELECT NR_IDFR_RCM, NR_IDFR_PROJ, NR_VRS_PRPT
                FROM {oracle_schema}.RCM_VRS
                GROUP BY NR_IDFR_RCM, NR_IDFR_PROJ, NR_VRS_PRPT
                HAVING COUNT(1) > 1
            )
        """,
        "AVS_SELD.JSON_INVALIDO": f"""
            SELECT COUNT(1) AS QTD
            FROM {oracle_schema}.AVS_SELD
            WHERE JS_AVS_PDRO IS NOT JSON
        """,
        "RCM_VRS.HEADER_INVALIDO": f"""
            SELECT COUNT(1) AS QTD
            FROM {oracle_schema}.RCM_VRS
            WHERE TX_PRPT_PRM_HDR_API IS NOT JSON
        """,
        "AVS_SELD.CATALOGO_ORFAO": f"""
            SELECT COUNT(1) AS QTD
            FROM {oracle_schema}.AVS_SELD avs
            WHERE NOT EXISTS (
                    SELECT 1 FROM {oracle_schema}.PBCO_CADD pbco
                    WHERE pbco.NR_IDFR_PBCO = avs.NR_IDFR_PBCO
                )
               OR NOT EXISTS (
                    SELECT 1 FROM {oracle_schema}.SGT_CADD sgt
                    WHERE sgt.NR_IDFR_SGT = avs.NR_IDFR_SGT
                )
               OR NOT EXISTS (
                    SELECT 1 FROM {oracle_schema}.APSC_TCN apsc
                    WHERE apsc.NR_IDFR_APSC = avs.NR_IDFR_APSC
                )
               OR NOT EXISTS (
                    SELECT 1 FROM {oracle_schema}.CNR_TCN cnr
                    WHERE cnr.NR_IDFR_CNR = avs.NR_IDFR_CNR
                )
               OR NOT EXISTS (
                    SELECT 1 FROM {oracle_schema}.TND_CGTV_TCN tnd
                    WHERE tnd.NR_IDFR_TND = avs.NR_IDFR_TND
                )
        """,
        "RCM_VLDD.PROJETO_ORFAO": f"""
            SELECT COUNT(1) AS QTD
            FROM {oracle_schema}.RCM_VLDD vldd
            WHERE NOT EXISTS (
                SELECT 1 FROM {oracle_schema}.PROJ_CADD proj
                WHERE proj.NR_IDFR_PROJ = vldd.NR_IDFR_PROJ
            )
        """,
        "RCM_VRS.RECOMENDACAO_ORFA": f"""
            SELECT COUNT(1) AS QTD
            FROM {oracle_schema}.RCM_VRS vrs
            WHERE NOT EXISTS (
                SELECT 1 FROM {oracle_schema}.RCM_VLDD vldd
                WHERE vldd.NR_IDFR_RCM = vrs.NR_IDFR_RCM
                  AND vldd.NR_IDFR_PROJ = vrs.NR_IDFR_PROJ
            )
        """,
        "RCM_VRS.AVISO_ORFAO": f"""
            SELECT COUNT(1) AS QTD
            FROM {oracle_schema}.RCM_VRS vrs
            WHERE NOT EXISTS (
                SELECT 1 FROM {oracle_schema}.AVS_SELD avs
                WHERE avs.NR_AVS_SELD = vrs.NR_AVS_SELD
            )
        """,
        "RCM_VLDD.VERSAO_PROPOSTA_AUSENTE": f"""
            SELECT COUNT(1) AS QTD
            FROM {oracle_schema}.RCM_VLDD vldd
            WHERE NOT EXISTS (
                SELECT 1 FROM {oracle_schema}.RCM_VRS vrs
                WHERE vrs.NR_IDFR_RCM = vldd.NR_IDFR_RCM
                  AND vrs.NR_IDFR_PROJ = vldd.NR_IDFR_PROJ
                  AND vrs.NR_VRS_PRPT = vldd.NR_VRS_PRPT
            )
        """,
        "RCM_VLDD.VERSAO_VALIDA_NAO_APROVADA": f"""
            SELECT COUNT(1) AS QTD
            FROM {oracle_schema}.RCM_VLDD vldd
            WHERE vldd.NR_VLDO_VRS > 0
              AND NOT EXISTS (
                  SELECT 1 FROM {oracle_schema}.RCM_VRS vrs
                  WHERE vrs.NR_IDFR_RCM = vldd.NR_IDFR_RCM
                    AND vrs.NR_IDFR_PROJ = vldd.NR_IDFR_PROJ
                    AND vrs.NR_VRS_PRPT = vldd.NR_VLDO_VRS
                    AND vrs.CD_EST_RCM_VRS = 2
              )
        """,
    }
    for verificacao, consulta in consultas_integridade.items():
        quantidade = int(cliente_oracle.run_select(consulta).collect()[0]["QTD"])
        status = "OK" if quantidade == 0 else "ERRO"
        print(f"[{status}] {verificacao}: inconsistências={quantidade}")
        if quantidade != 0:
            inconsistencias.append((verificacao, 0, quantidade))

consulta_versoes = f"""
SELECT COUNT(1) AS QTD
FROM {oracle_schema}.PMPT_TCN
WHERE NR_VRS_ATU <> 1
"""
prompts_fora_da_versao = int(
    cliente_oracle.run_select(consulta_versoes).collect()[0]["QTD"]
)
if prompts_fora_da_versao != 0:
    inconsistencias.append(("PMPT_TCN.NR_VRS_ATU", 0, prompts_fora_da_versao))

ids_projeto = [
    int(linha["NR_IDFR_PROJ"])
    for linha in cliente_oracle.run_select(
        f"SELECT NR_IDFR_PROJ FROM {oracle_schema}.PROJ_CADD"
    ).collect()
]
status_projeto = "OK" if ids_projeto == [ID_PROJETO_PADRAO] else "ERRO"
print(
    f"[{status_projeto}] PROJ_CADD.NR_IDFR_PROJ: "
    f"esperado={[ID_PROJETO_PADRAO]} Oracle={ids_projeto}"
)
if ids_projeto != [ID_PROJETO_PADRAO]:
    inconsistencias.append(
        ("PROJ_CADD.NR_IDFR_PROJ", [ID_PROJETO_PADRAO], ids_projeto)
    )

if inconsistencias:
    raise RuntimeError(f"Verificação final falhou: {inconsistencias}")

print("Verificação final concluída: tabelas e versões estão coerentes.")


## 08. Encerramento


In [ ]:
%spark cleanup
